##### debug

In [1]:
import os 
import sys
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

## Spark Session

In [46]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder.master('local[*]').getOrCreate()

## Reading data

In [128]:
df_vehicles = spark.read.option('multiLine','true').json('../data/sample_vehicles.json')

df_traffic = spark.read.option('multiLine','true').json('../data/sample_traffic.json')

df_stops= spark.read.option('multiLine','true').json('../data/sample_stops.json')

In [129]:
df_vehicles.show()

+--------------------+--------------------+
|               buses|               trams|
+--------------------+--------------------+
|[{2, 52.222349, 2...|[{016, 52.25597, ...|
+--------------------+--------------------+



#### Some variables

In [130]:
LAT_MIN, LAT_MAX = 52.05, 52.40
LON_MIN, LON_MAX = 20.70, 21.45

In [205]:
OUTPUT_DIR = '../data/cleaned_data'

## Transformations

### Busses/Trams

In [131]:
buses_df = df_vehicles.select(F.explode('buses').alias('buses'),)
buses_df = buses_df.select('buses.*')
buses_df = buses_df.withColumn('Type',F.lit('Bus'))

trams_df = df_vehicles.select(F.explode('trams').alias('trams'))
trams_df = trams_df.select('trams.*')
trams_df = trams_df.withColumn('Type',F.lit('Tram'))

vehicle_df = buses_df.unionByName(trams_df)


In [132]:
df_stops.show(1)

+--------------------+
|              values|
+--------------------+
|[{zespol, 4085}, ...|
+--------------------+
only showing top 1 row



In [133]:
vehicle_df.select('*').count()

1406

In [208]:
vehicle_df.show(5)

+-------+----------+-----+----------+-------------------+-------------+----+
|Brigade|       Lat|Lines|       Lon|               Time|VehicleNumber|Type|
+-------+----------+-----+----------+-------------------+-------------+----+
|    697|52.0762783|  L-5|21.0278965|2026-07-07 22:42:03|        15077| Bus|
|      2|52.3976365|  L41|20.9409613|2026-07-07 22:42:04|        15078| Bus|
|      2|52.3399126|  L49|20.9468391|2026-07-07 22:42:05|        15079| Bus|
|      2|52.3918346|  L34| 20.924858|2026-07-07 22:42:03|        15082| Bus|
|    500| 52.394281|  L-9|20.9315668|2026-07-07 22:42:04|        15087| Bus|
+-------+----------+-----+----------+-------------------+-------------+----+
only showing top 5 rows



In [135]:
vehicle_df.select([
    F.sum(F.when((F.col(c).isNull()) | (F.col(c)==''),1).otherwise(0)).alias(c) for c in vehicle_df.columns
]).show()

+-------+---+-----+---+----+-------------+----+
|Brigade|Lat|Lines|Lon|Time|VehicleNumber|Type|
+-------+---+-----+---+----+-------------+----+
|      0|  0|    0|  0|   0|            0|   0|
+-------+---+-----+---+----+-------------+----+



In [136]:
vehicle_df.schema

StructType([StructField('Brigade', StringType(), True), StructField('Lat', DoubleType(), True), StructField('Lines', StringType(), True), StructField('Lon', DoubleType(), True), StructField('Time', StringType(), True), StructField('VehicleNumber', StringType(), True), StructField('Type', StringType(), False)])

In [137]:
vehicle_df = (
    vehicle_df
    .withColumn('Time',F.to_timestamp(F.col('Time'),'yyyy-MM-dd HH:mm:ss'))
)

In [138]:
vehicle_df =vehicle_df.filter(
    (F.col('Lat') >= LAT_MIN) & (F.col('Lat') <= LAT_MAX) &
    (F.col("Lon") >=LON_MIN) & (F.col('Lon') <= LON_MAX)
)

In [139]:
vehicle_df=vehicle_df.filter(
    (F.col('Time') >= F.current_timestamp()-F.expr('INTERVAL 2 HOURS'))
)

In [209]:
vehicle_df=(
    vehicle_df
    .withColumn('year',F.date_format(F.col('Time'),'yyyy'))
    .withColumn('month',F.date_format(F.col('Time'),'MM'))
    .withColumn('day',F.date_format(F.col('Time'),'dd'))
    .withColumn('hour',F.date_format(F.col('Time'),'HH'))
)

In [214]:
vehicle_df.write.mode('append').partitionBy('year','month','day','hour').parquet(f'{OUTPUT_DIR}/vehicles')

### Stops

In [140]:
stops_df = df_stops.withColumn('id',F.monotonically_increasing_id())
stops_df = stops_df.select('id',F.explode('values').alias('item'))
stops_df=stops_df.select('id',F.col('item.key').alias('key'),F.col('item.value').alias('value'))

In [141]:
stops_df.show(7)

+---+-------------+---------+
| id|          key|    value|
+---+-------------+---------+
|  0|       zespol|     4085|
|  0|       slupek|       02|
|  0|nazwa_zespolu| Radarowa|
|  0|     id_ulicy|     2316|
|  0|     szer_geo|52.188762|
|  0|     dlug_geo|20.962959|
|  1|       zespol|     4085|
+---+-------------+---------+
only showing top 7 rows



In [142]:
stops_df=stops_df.groupBy('id').pivot('key').agg(F.first('value'))

In [143]:
stops_df = (stops_df
    .withColumn('lat',F.col('szer_geo').cast('Double'))
    .withColumn('lon',F.col('dlug_geo').cast('Double'))
    .filter(
        (F.col("lat").between(LAT_MIN,LAT_MAX)) &
        (F.col('lon').between(LON_MIN,LON_MAX))
    )
    .select(
        F.col('id_ulicy'),
        F.col('nazwa_zespolu'),
        F.col('slupek'),
        F.col("zespol"),
        F.col('lat'),
        F.col('lon')
    )
)

In [144]:
stops_df.show(3)

+--------+-------------+------+------+---------+---------+
|id_ulicy|nazwa_zespolu|slupek|zespol|      lat|      lon|
+--------+-------------+------+------+---------+---------+
|    2316|     Radarowa|    02|  4085|52.188762|20.962959|
|    2316|     Radarowa|    01|  4085|52.188116|20.964077|
|    2237|        Hynka|    01|  4011|52.190665|20.958908|
+--------+-------------+------+------+---------+---------+
only showing top 3 rows



In [213]:
stops_df.write.mode('overwrite').parquet(f'{OUTPUT_DIR}/stops')

### Traffic

In [199]:
traffic_df = df_traffic.withColumn(
    'id',F.monotonically_increasing_id()
)

In [200]:
traffic_df = (
    traffic_df
    .select('*',
    F.explode('geo').alias('geo_group'))
)

In [201]:
traffic_df = (
    traffic_df
    .select('*',F.explode('geo_group').alias('geomerty'))
)

In [202]:
traffic_df=traffic_df.select(
    '*',
    F.col('geomerty.geomType').alias('geoType'),
    F.col('geomerty.latlngs').alias('latlngs')
)

In [ ]:
traffic_df = (
    traffic_df
    .select('*',F.posexplode('latlngs').alias('point_order','point'))
    .select('*',F.col('point.lat').alias('lat'),
                F.col('point.lng').alias('lon')) 
).drop('geo_group','geomerty','latlngs','point','geo')

In [215]:
traffic_df.show(2)

+--------------------+------------------+-------------------+--------------------+---+--------------------+-------------------+-----------+-------+-----------+-----------------+------------------+
|             content|       detour_type|           end_date|                 geo| id|                name|         start_date|    streets|geoType|point_order|              lat|               lon|
+--------------------+------------------+-------------------+--------------------+---+--------------------+-------------------+-----------+-------+-----------+-----------------+------------------+
|[Do końca lipca p...|[Zamknięcie ulicy]|2026-07-31 19:00:00|[[{1, [{52.241403...|  0|Wyremontują Wierz...|2026-07-08 06:00:00|[Wierzynka]|      1|          0|52.24140396706791|20.966913700103763|
|[Do końca lipca p...|[Zamknięcie ulicy]|2026-07-31 19:00:00|[[{1, [{52.241403...|  0|Wyremontują Wierz...|2026-07-08 06:00:00|[Wierzynka]|      1|          1|52.24142367601795|20.967198014259342|
+--------------

In [ ]:
traffic_df.write.mode('overwrite').parquet(f'{OUTPUT_DIR}/traffic')